# XQuality No-Gold Evaluation After Ablation

This notebook is meant to be run **after** the NeoOLAF ablation notebook has produced Layer 12 outputs. It runs the improved no-gold evaluator over XQuality results and runs a field-aware multi-judge LLM panel by default.

The no-gold evaluator checks structural validity, ontology health, source grounding, structured table-record support, legacy faithfulness, BLEU-style overlap, optional ontology alignment, and a multi-judge panel with blue support, red critic, profile judge, and arbiter.

The panel now reports `inconclusive` separately from `invalid` when a low-agreement panel decision conflicts with deterministic source-grounding and XQuality field mapping.

## Multi-judge LLM panel

This notebook uses the no-gold automatic metrics plus an optional multi-judge LLM panel enabled by default. The panel contains a blue support judge, a red critic judge, an XQuality profile judge, and a final arbiter. It is field-aware for the XQuality mapping: `TRIGGERS = cause -> alarm`, `CAUSES = alarm -> effet`, `REQUIRES = alarm -> intervention`, `HANDLED_BY = alarm -> responsible`, and `REFERENCES = alarm -> technical reference`.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import json
import pandas as pd
import subprocess
import sys


def find_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "src" / "neoolaf").exists():
            return p
    raise RuntimeError("Could not find NeoOLAF root. Run this notebook inside the NeoOLAF repository.")

ROOT = find_root()
print("ROOT:", ROOT)

# Ensure src/ import when running from notebook.
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

ROOT: c:\Users\henri\Documents\git\post-doc\NeoOLAF


In [3]:
# Input PDF and gold truth paths, kept here for consistency with the ablation notebook.
PDF_PATH = ROOT / "data" / "XQuality" / "Textual" / "Chapitre_8_Alarmes_et_messages.pdf"
GOLD_JSON = ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"
GOLD_EXCEL = ROOT / "data" / "XQuality" / "Machine32" / "machine32_triples.xlsx"
SEED_ONTOLOGY = ROOT / "data" / "ontology" / "ContextOntology-COInd4.owl"

# Main run folders.
RUNS_ROOT = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32"
LAYER12_RUN_DIR = RUNS_ROOT / "layer12_from_l11"
LAYER12_STATE = LAYER12_RUN_DIR / "layer12_serialization" / "state.json"
EXPORT_DIR = LAYER12_RUN_DIR / "exports"

EVALUATIONS_ROOT = RUNS_ROOT / "evaluations"
NO_GOLD_EVAL_DIR = EVALUATIONS_ROOT / "no_gold_layer12"
GOLD_EVAL_DIR = EVALUATIONS_ROOT / "gold_layer12"

MODEL = "openrouter/openai/gpt-oss-20b"
DOCUMENT_PROFILE = "xquality_relaxed_recall"

print("LAYER12_STATE:", LAYER12_STATE)
print("EXPORT_DIR:", EXPORT_DIR)
print("NO_GOLD_EVAL_DIR:", NO_GOLD_EVAL_DIR)

LAYER12_STATE: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer12_from_l11\layer12_serialization\state.json
EXPORT_DIR: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer12_from_l11\exports
NO_GOLD_EVAL_DIR: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\evaluations\no_gold_layer12


## 1. Run improved automatic no-gold evaluation

This does not use the manual gold truth. It uses the saved Layer 12 state, provenance, full source chunks, structured source records, ontology hints, and optional seed ontology alignment.

In [6]:
NO_GOLD_EVAL_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-m", "neoolaf.evaluation", "no-gold",
    "--state", str(LAYER12_STATE),
    "--reference-ontology", str(SEED_ONTOLOGY),
    "--output", str(NO_GOLD_EVAL_DIR),
]

print(" ".join(cmd))
result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("No-gold evaluation failed")

c:\Users\henri\Documents\git\post-doc\neoolafvenv\Scripts\python.exe -m neoolaf.evaluation no-gold --state c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer12_from_l11\layer12_serialization\state.json --reference-ontology c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\ontology\ContextOntology-COInd4.owl --output c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\evaluations\no_gold_layer12



In [7]:
summary_path = NO_GOLD_EVAL_DIR / "no_gold.summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

print(json.dumps({
    "validation": summary.get("validation", {}),
    "source_grounding": summary.get("source_grounding", {}),
    "faithfulness": summary.get("faithfulness", {}),
    "bleu": summary.get("bleu", {}),
    "ontology_alignment": summary.get("ontology_alignment", {}),
}, indent=2, ensure_ascii=False)[:5000])

{
  "validation": {
    "is_valid": true,
    "issues": {
      "total": 0,
      "errors": 0,
      "warnings": 0,
      "by_type": {}
    },
    "graph_health": {
      "candidate_triples": 226,
      "inferred_triples": 226,
      "dedup_ratio": 1.0,
      "avg_triple_confidence": 1.0
    },
    "completions": {
      "total": 2999,
      "by_type": {
        "rdf_type_completion": 304,
        "candidate_concept_alignment_completion": 304,
        "skos_pref_label_completion": 304,
        "rdfs_comment_completion": 304,
        "skos_alt_label_completion": 1783
      },
      "avg_confidence": 0.9108202734244749
    },
    "ontology_health": {
      "total_concepts": 304,
      "total_relations": 5,
      "total_axioms": 628,
      "orphan_concept_ratio": 0.0,
      "domain_range_coverage": 1.0
    }
  },
  "source_grounding": {
    "total_triples": 226,
    "provenance_coverage": 1.0,
    "relation_marker_support_rate": 1.0,
    "endpoint_support_rate": 1.0,
    "table_record_sup

In [8]:
source_csv = NO_GOLD_EVAL_DIR / "no_gold.source_grounding.per_triple.csv"
source_df = pd.read_csv(source_csv)
print(source_df.shape)
display(source_df.head(20))

(226, 15)


,triple_id,predicate,chunk_id,has_provenance,relation_marker_supported,endpoint_supported,table_record_supported,structured_record_supported,source_grounded,support_score,support_mode,subject_label,object_label,evidence_preview,notes
0,triple_00000,TRIGGERS,table_p0003_8_1,True,True,True,True,True,True,1.0,structured_record:cause_to_alarm,Indicates that the punch button has been pressed,EMERGENCY ACTIVE,Alarme n°: 1001 texte: URGENCE ACTIVE type: St...,['structured_record:cause_to_alarm']
1,triple_00001,CAUSES,table_p0003_8_1,True,True,True,True,True,True,1.0,structured_record:alarm_to_effect,EMERGENCY ACTIVE,"Alarm with immediate and controlled stop, open...",Alarme n°: 1001 texte: URGENCE ACTIVE type: St...,['structured_record:alarm_to_effect']
2,triple_00002,REQUIRES,table_p0003_8_1,True,True,True,True,True,True,1.0,structured_record:alarm_to_intervention,EMERGENCY ACTIVE,Release the button and press the machine start...,Alarme n°: 1001 texte: URGENCE ACTIVE type: St...,['structured_record:alarm_to_intervention']
3,triple_00003,HANDLED_BY,table_p0003_8_1,True,True,True,True,True,True,1.0,structured_record:alarm_to_responsible,EMERGENCY ACTIVE,Operator,Alarme n°: 1001 texte: URGENCE ACTIVE type: St...,['structured_record:alarm_to_responsible']
4,triple_00004,HANDLED_BY,table_p0003_8_1,True,True,True,True,True,True,1.0,structured_record:alarm_to_responsible,EMERGENCY ACTIVE,Maintenance Technician,Alarme n°: 1001 texte: URGENCE ACTIVE type: St...,['structured_record:alarm_to_responsible']
5,triple_00005,REFERENCES,table_p0003_8_1,True,True,True,True,True,True,1.0,structured_record:alarm_to_reference,EMERGENCY ACTIVE,"See page 5 of the electrical diagram, input X8.4",Alarme n°: 1001 texte: URGENCE ACTIVE type: St...,['structured_record:alarm_to_reference']
6,triple_00006,TRIGGERS,table_p0003_8_2,True,True,True,True,True,True,1.0,structured_record:cause_to_alarm,Indicates that the power circuits were not act...,MACHINE OFFLINE,Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...,['structured_record:cause_to_alarm']
7,triple_00007,CAUSES,table_p0003_8_2,True,True,True,True,True,True,1.0,structured_record:alarm_to_effect,MACHINE OFFLINE,"Alarm with immediate and controlled stop, open...",Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...,['structured_record:alarm_to_effect']
8,triple_00008,REQUIRES,table_p0003_8_2,True,True,True,True,True,True,1.0,structured_record:alarm_to_intervention,MACHINE OFFLINE,Press the machine start button.,Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...,['structured_record:alarm_to_intervention']
9,triple_00009,REFERENCES,table_p0003_8_2,True,True,True,True,True,True,1.0,structured_record:alarm_to_reference,MACHINE OFFLINE,"See page 5 of the electrical diagram, input X2.0.",Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...,['structured_record:alarm_to_reference']


In [9]:
# Show weakest triples according to automatic source grounding.
weak_cols = [
    "triple_id", "predicate", "support_score", "source_grounded", "support_mode",
    "subject_label", "object_label", "notes", "evidence_preview",
]
weak_df = source_df.sort_values(["source_grounded", "support_score"], ascending=[True, True])
display(weak_df[weak_cols].head(30))

,triple_id,predicate,support_score,source_grounded,support_mode,subject_label,object_label,notes,evidence_preview
0,triple_00000,TRIGGERS,1.0,True,structured_record:cause_to_alarm,Indicates that the punch button has been pressed,EMERGENCY ACTIVE,['structured_record:cause_to_alarm'],Alarme n°: 1001 texte: URGENCE ACTIVE type: St...
1,triple_00001,CAUSES,1.0,True,structured_record:alarm_to_effect,EMERGENCY ACTIVE,"Alarm with immediate and controlled stop, open...",['structured_record:alarm_to_effect'],Alarme n°: 1001 texte: URGENCE ACTIVE type: St...
2,triple_00002,REQUIRES,1.0,True,structured_record:alarm_to_intervention,EMERGENCY ACTIVE,Release the button and press the machine start...,['structured_record:alarm_to_intervention'],Alarme n°: 1001 texte: URGENCE ACTIVE type: St...
3,triple_00003,HANDLED_BY,1.0,True,structured_record:alarm_to_responsible,EMERGENCY ACTIVE,Operator,['structured_record:alarm_to_responsible'],Alarme n°: 1001 texte: URGENCE ACTIVE type: St...
4,triple_00004,HANDLED_BY,1.0,True,structured_record:alarm_to_responsible,EMERGENCY ACTIVE,Maintenance Technician,['structured_record:alarm_to_responsible'],Alarme n°: 1001 texte: URGENCE ACTIVE type: St...
5,triple_00005,REFERENCES,1.0,True,structured_record:alarm_to_reference,EMERGENCY ACTIVE,"See page 5 of the electrical diagram, input X8.4",['structured_record:alarm_to_reference'],Alarme n°: 1001 texte: URGENCE ACTIVE type: St...
6,triple_00006,TRIGGERS,1.0,True,structured_record:cause_to_alarm,Indicates that the power circuits were not act...,MACHINE OFFLINE,['structured_record:cause_to_alarm'],Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...
7,triple_00007,CAUSES,1.0,True,structured_record:alarm_to_effect,MACHINE OFFLINE,"Alarm with immediate and controlled stop, open...",['structured_record:alarm_to_effect'],Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...
8,triple_00008,REQUIRES,1.0,True,structured_record:alarm_to_intervention,MACHINE OFFLINE,Press the machine start button.,['structured_record:alarm_to_intervention'],Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...
9,triple_00009,REFERENCES,1.0,True,structured_record:alarm_to_reference,MACHINE OFFLINE,"See page 5 of the electrical diagram, input X2.0.",['structured_record:alarm_to_reference'],Alarme n°: 1002 texte: MACHINE HORS CIRCUIT ty...


## 2. LLM-as-a-judge, enabled by default

This cell runs the profile-aware LLM judge automatically. It does **not** use gold truth. The judge receives the source evidence, the predicted triple, and the XQuality relation semantics used by NeoOLAF.

By default, it judges weak/suspicious triples first. If no weak triples remain after the improved source-grounding evaluator, it judges a deterministic sample.

Set `ENABLE_LLM_JUDGE = False` only if you want to avoid API calls.


The judge now runs in parallel by default (`JUDGE_MAX_WORKERS = 4`) and applies deterministic XQuality field-aware correction after the LLM answer, so CAUSES/TRIGGERS are judged according to the table-field profile rather than generic causal intuition.


Recoverable subjudge parse failures are not counted by default in the final panel summary. Set `COUNT_SUBJUDGE_PARSE_ERRORS = True` only if you want to audit provider/parser instability.


In [4]:
# Multi-judge LLM panel evaluation.
# This is enabled by default because it is part of the no-gold evaluation protocol.
# It may call your provider through LiteLLM/OpenRouter.

import os

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass

ENABLE_LLM_JUDGE = True
JUDGE_MODEL = MODEL
JUDGE_MAX_ITEMS = 25
JUDGE_ONLY_WEAK = True
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS = 1200
JUDGE_MAX_WORKERS = 4  # parallel panel calls; each triple uses blue, red, profile and arbiter judges
COUNT_SUBJUDGE_PARSE_ERRORS = False  # default: do not count recoverable subjudge parse failures in final parse_error_count

LLM_JUDGE_EVAL_DIR = EVALUATIONS_ROOT / "no_gold_layer12_llm_judge_panel"

assert LAYER12_STATE.exists(), f"Layer 12 state not found: {LAYER12_STATE}"
assert SEED_ONTOLOGY.exists(), f"Seed ontology not found: {SEED_ONTOLOGY}"

if ENABLE_LLM_JUDGE:
    LLM_JUDGE_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "-m", "neoolaf.evaluation", "no-gold",
        "--state", str(LAYER12_STATE),
        "--reference-ontology", str(SEED_ONTOLOGY),
        "--output", str(LLM_JUDGE_EVAL_DIR),
        "--llm-judge-panel",
        "--judge-model", JUDGE_MODEL,
        "--judge-max-items", str(JUDGE_MAX_ITEMS),
        "--judge-temperature", str(JUDGE_TEMPERATURE),
        "--judge-max-tokens", str(JUDGE_MAX_TOKENS),
        "--judge-max-workers", str(JUDGE_MAX_WORKERS),
    ]
    if not JUDGE_ONLY_WEAK:
        cmd.append("--judge-all")
    if COUNT_SUBJUDGE_PARSE_ERRORS:
        cmd.append("--count-subjudge-parse-errors")

    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        cwd=ROOT,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )

    print("\nSTDOUT:")
    print(result.stdout)

    if result.stderr.strip():
        print("\nSTDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("LLM judge evaluation failed")
else:
    print("LLM judge disabled. Set ENABLE_LLM_JUDGE = True to run it.")


Running command:
c:\Users\henri\Documents\git\post-doc\neoolafvenv\Scripts\python.exe -m neoolaf.evaluation no-gold --state c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer12_from_l11\layer12_serialization\state.json --reference-ontology c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\ontology\ContextOntology-COInd4.owl --output c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\evaluations\no_gold_layer12_llm_judge_panel --llm-judge-panel --judge-model openrouter/openai/gpt-oss-20b --judge-max-items 25 --judge-temperature 0.0 --judge-max-tokens 1200 --judge-max-workers 4

STDOUT:


STDERR:
13:43:25 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape â€” Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
13:43:25 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime

In [5]:
if ENABLE_LLM_JUDGE:
    with open(LLM_JUDGE_EVAL_DIR / "no_gold.summary.json", "r", encoding="utf-8") as f:
        judge_summary = json.load(f)

    panel_summary = judge_summary.get("llm_judge_panel", {})
    print(json.dumps(panel_summary, indent=2, ensure_ascii=False))
    print(
        "Panel counts: "
        f"valid={panel_summary.get('valid_count', 0)}, "
        f"weak={panel_summary.get('weak_count', 0)}, "
        f"invalid={panel_summary.get('invalid_count', 0)}, "
        f"inconclusive={panel_summary.get('inconclusive_count', 0)}"
    )

    judge_csv = LLM_JUDGE_EVAL_DIR / "no_gold.llm_judge_panel.per_triple.csv"
    judge_df = pd.read_csv(judge_csv)
    print("Judged rows:", len(judge_df))
    if COUNT_SUBJUDGE_PARSE_ERRORS and "parse_error" in judge_df.columns:
        print("Subjudge parse errors:", int(judge_df["parse_error"].fillna("").astype(str).str.len().gt(0).sum()))
    elif "parse_error_count" in panel_summary:
        print("Final panel parse_error_count:", panel_summary.get("parse_error_count", 0))

    sort_col = "final_score" if "final_score" in judge_df.columns else "score"
    display(judge_df.sort_values(sort_col).head(30))

    source_csv = LLM_JUDGE_EVAL_DIR / "no_gold.source_grounding.per_triple.csv"
    if source_csv.exists():
        source_df = pd.read_csv(source_csv)
        if "triple_id" in source_df.columns and "triple_id" in judge_df.columns:
            compare_df = source_df.merge(
                judge_df,
                on="triple_id",
                how="inner",
                suffixes=("_auto", "_judge"),
            )
            verdict_col = "final_verdict" if "final_verdict" in compare_df.columns else "verdict"
            score_col = "final_score" if "final_score" in compare_df.columns else "score"
            auto_source_col = "source_grounded_auto" if "source_grounded_auto" in compare_df.columns else "source_grounded"

            recovered = compare_df[
                (compare_df[auto_source_col] == False)
                & (compare_df[verdict_col].astype(str).str.lower() == "valid")
            ]
            print("Auto weak but panel valid:", len(recovered))
            display(recovered.head(30))

            uncertain = compare_df[
                (compare_df[verdict_col].astype(str).str.lower().isin(["weak", "invalid", "inconclusive"]))
                | (compare_df[score_col] < 0.75)
                | (compare_df.get("agreement", pd.Series([], dtype=str)).astype(str).str.lower() == "low")
            ]
            print("Panel weak/invalid/inconclusive/low-agreement cases:", len(uncertain))
            display(uncertain.head(30))
else:
    print("LLM judge panel disabled.")


{
  "model": "openrouter/openai/gpt-oss-20b",
  "judged_count": 25,
  "valid_count": 24,
  "weak_count": 1,
  "invalid_count": 0,
  "inconclusive_count": 0,
  "parse_error_count": 0,
  "count_subjudge_parse_errors": false,
  "average_score": 0.944,
  "supported_rate": 1.0,
  "relation_supported_rate": 1.0,
  "direction_correct_rate": 1.0,
  "high_agreement_count": 24,
  "medium_agreement_count": 0,
  "low_agreement_count": 1
}
Panel counts: valid=24, weak=1, invalid=0, inconclusive=0
Judged rows: 25
Final panel parse_error_count: 0


,triple_id,predicate,subject_label,object_label,chunk_id,source_grounded,automatic_support_score,final_verdict,final_score,agreement,...,relation_supported,direction_correct,over_specific,missing_split,rationale,parse_error,blue_raw_response,red_raw_response,profile_raw_response,arbiter_raw_response
20,triple_00020,REQUIRES,OPEN SIDE PROTECTIONS,"Verify that the side protections are closed, c...",table_p0004_8_5,True,1.0,weak,0.80,low,...,True,True,True,True,Object merges actions not in source.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":f..."
0,triple_00000,TRIGGERS,Indicates that the punch button has been pressed,EMERGENCY ACTIVE,table_p0003_8_1,True,1.0,valid,0.95,high,...,True,True,False,False,Field mapping supports the relation.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t..."
22,triple_00022,TRIGGERS,A thermal switch of the auxiliary motors was t...,Thermo-magnetic switches. Stop at end of cycle,table_p0004_8_6,True,1.0,valid,0.95,high,...,True,True,False,False,Cause matches alarm label per XQuality mapping.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t..."
21,triple_00021,REFERENCES,OPEN SIDE PROTECTIONS,"See page 7 of the electrical diagram, input X2.5",table_p0004_8_5,True,1.0,valid,0.95,high,...,True,True,False,False,Reference marker matches alarm and page.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t..."
19,triple_00019,CAUSES,OPEN SIDE PROTECTIONS,Alarm with immediate controlled stop and openi...,table_p0004_8_5,True,1.0,valid,0.95,high,...,True,True,False,False,Field mapping supports the alarm -> effect rel...,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t..."
18,triple_00018,TRIGGERS,The side protections are open.,OPEN SIDE PROTECTIONS,table_p0004_8_5,True,1.0,valid,0.95,high,...,True,True,False,False,Field mapping supports the relation.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t..."
17,triple_00017,REFERENCES,SLIDING DOOR UNLOCKED,Page 19 — input X3.2,table_p0004_8_4,True,1.0,valid,0.95,high,...,True,True,False,False,Field mapping supports the relation.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t..."
16,triple_00016,REQUIRES,SLIDING DOOR UNLOCKED,Release the unlocking button of the door befor...,table_p0004_8_4,True,1.0,valid,0.95,high,...,True,True,False,False,Field mapping supports the relation.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t..."
15,triple_00015,CAUSES,SLIDING DOOR UNLOCKED,Alarm with immediate and controlled stop,table_p0004_8_4,True,1.0,valid,0.95,high,...,True,True,False,False,Field mapping supports the relation.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_suppor

Auto weak but panel valid: 0


,triple_id,predicate_auto,chunk_id_auto,has_provenance,relation_marker_supported,endpoint_supported,table_record_supported,structured_record_supported,source_grounded_auto,support_score,...,relation_supported,direction_correct,over_specific,missing_split,rationale,parse_error,blue_raw_response,red_raw_response,profile_raw_response,arbiter_raw_response


Panel weak/invalid/inconclusive/low-agreement cases: 1


,triple_id,predicate_auto,chunk_id_auto,has_provenance,relation_marker_supported,endpoint_supported,table_record_supported,structured_record_supported,source_grounded_auto,support_score,...,relation_supported,direction_correct,over_specific,missing_split,rationale,parse_error,blue_raw_response,red_raw_response,profile_raw_response,arbiter_raw_response
20,triple_00020,REQUIRES,table_p0004_8_5,True,True,True,True,True,True,1.0,...,True,True,True,True,Object merges actions not in source.,NaN,"{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":t...","{""subject_supported"":true,""object_supported"":f..."


## 3. Optional gold evaluation for the same Layer 12 export

This is only for comparison with the available XQuality gold truth. It is not required for no-gold evaluation.

In [10]:
RUN_GOLD_EVAL = True

if RUN_GOLD_EVAL:
    GOLD_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "-m", "neoolaf.evaluation", "evaluate",
        "--dataset", "xquality",
        "--method", "neoolaf",
        "--profile", DOCUMENT_PROFILE,
        "--input", str(EXPORT_DIR),
        "--gold", str(GOLD_JSON),
        "--ontology-path", str(SEED_ONTOLOGY),
        "--output", str(GOLD_EVAL_DIR),
    ]
    print(" ".join(cmd))
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Gold evaluation failed")
else:
    print("Gold evaluation disabled.")

c:\Users\henri\Documents\git\post-doc\neoolafvenv\Scripts\python.exe -m neoolaf.evaluation evaluate --dataset xquality --method neoolaf --profile xquality_relaxed_recall --input c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer12_from_l11\exports --gold c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json --ontology-path c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\ontology\ContextOntology-COInd4.owl --output c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\evaluations\gold_layer12



In [11]:
if RUN_GOLD_EVAL:
    summary_candidates = list(GOLD_EVAL_DIR.rglob("metrics.summary.json"))
    print("Found summaries:", summary_candidates)
    if summary_candidates:
        with open(summary_candidates[0], "r", encoding="utf-8") as f:
            gold_summary = json.load(f)
        print(json.dumps(gold_summary, indent=2, ensure_ascii=False)[:5000])

    flat_candidates = list(GOLD_EVAL_DIR.rglob("metrics.flat.csv"))
    if flat_candidates:
        display(pd.read_csv(flat_candidates[0]))

    per_rel_candidates = list(GOLD_EVAL_DIR.rglob("per_relation_metrics.csv"))
    if per_rel_candidates:
        display(pd.read_csv(per_rel_candidates[0]))

Found summaries: [WindowsPath('c:/Users/henri/Documents/git/post-doc/NeoOLAF/examples/XQualityMachine32/runs/xquality_machine32/evaluations/gold_layer12/metrics.summary.json')]
{
  "dataset": "xquality",
  "method": "neoolaf",
  "profile": "xquality_relaxed_recall",
  "evaluation_protocol": {
    "extraction": "relaxed_gt_guided_domain_canonicalization",
    "validation": "validation_on_canonicalized_triples_old_style"
  },
  "entity": {
    "tp": 218,
    "fp": 275,
    "fn": 76,
    "precision": 0.4421906693711968,
    "recall": 0.7414965986394558,
    "f1": 0.554002541296061
  },
  "relation": {
    "tp": 193,
    "fp": 0,
    "fn": 246,
    "precision": 1.0,
    "recall": 0.4396355353075171,
    "f1": 0.6107594936708861
  },
  "per_relation": [
    {
      "relation": "CAUSES",
      "pred_count": 24,
      "gt_count": 105,
      "tp": 24,
      "fp": 0,
      "fn": 81,
      "precision": 1.0,
      "recall": 0.22857142857142856,
      "f1": 0.3720930232558139
    },
    {
      "r

,method,dataset,profile,entity_precision,entity_recall,entity_f1,relation_precision,relation_recall,relation_f1,STR,CR,PC,OC,CV,DVS
0,neoolaf,xquality,xquality_relaxed_recall,0.442191,0.741497,0.554003,1.0,0.439636,0.610759,1.0,0.0,1.0,1.0,0.015544,1.0


,relation,pred_count,gt_count,tp,fp,fn,precision,recall,f1
0,CAUSES,24,105,24,0,81,1.0,0.228571,0.372093
1,HANDLED_BY,12,96,12,0,84,1.0,0.125000,0.222222
2,REFERENCES,20,23,20,0,3,1.0,0.869565,0.930233
3,REQUIRES,83,132,83,0,49,1.0,0.628788,0.772093
4,TRIGGERS,54,83,54,0,29,1.0,0.650602,0.788321


## 4. Batch no-gold evaluation over all Layer 12 ablation runs

This scans the ablation folder for every `layer12_serialization/state.json` and evaluates each run in a separate output folder.

In [12]:
RUN_BATCH_NO_GOLD = True
BATCH_NO_GOLD_ROOT = EVALUATIONS_ROOT / "batch_no_gold"

layer12_states = sorted(RUNS_ROOT.rglob("layer12_serialization/state.json"))
print("Found Layer 12 states:", len(layer12_states))
for p in layer12_states:
    print("-", p.relative_to(RUNS_ROOT))

if RUN_BATCH_NO_GOLD:
    rows = []
    for state_path in layer12_states:
        run_name = state_path.parent.parent.name
        out_dir = BATCH_NO_GOLD_ROOT / run_name
        out_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            sys.executable, "-m", "neoolaf.evaluation", "no-gold",
            "--state", str(state_path),
            "--reference-ontology", str(SEED_ONTOLOGY),
            "--output", str(out_dir),
        ]
        result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
        if result.returncode != 0:
            print("FAILED", run_name)
            print(result.stderr)
            continue
        with open(out_dir / "no_gold.summary.json", "r", encoding="utf-8") as f:
            s = json.load(f)
        sg = s.get("source_grounding", {})
        val = s.get("validation", {})
        graph = val.get("graph_health", {})
        ont = val.get("ontology_health", {})
        rows.append({
            "run": run_name,
            "candidate_triples": graph.get("candidate_triples"),
            "validation_is_valid": val.get("is_valid"),
            "issues_total": val.get("issues", {}).get("total"),
            "source_grounding_rate": sg.get("source_grounding_rate"),
            "structured_record_support_rate": sg.get("structured_record_support_rate"),
            "average_support_score": sg.get("average_support_score"),
            "provenance_coverage": sg.get("provenance_coverage"),
            "domain_range_coverage": ont.get("domain_range_coverage"),
            "orphan_concept_ratio": ont.get("orphan_concept_ratio"),
        })

    batch_df = pd.DataFrame(rows).sort_values("source_grounding_rate", ascending=False)
    batch_csv = BATCH_NO_GOLD_ROOT / "batch_no_gold_summary.csv"
    batch_csv.parent.mkdir(parents=True, exist_ok=True)
    batch_df.to_csv(batch_csv, index=False)
    display(batch_df)
    print("Saved:", batch_csv)

Found Layer 12 states: 1
- layer12_from_l11\layer12_serialization\state.json


,run,candidate_triples,validation_is_valid,issues_total,source_grounding_rate,structured_record_support_rate,average_support_score,provenance_coverage,domain_range_coverage,orphan_concept_ratio
0,layer12_from_l11,226,True,0,1.0,1.0,1.0,1.0,1.0,0.0


Saved: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\evaluations\batch_no_gold\batch_no_gold_summary.csv


## 5. Batch LLM judge over selected ablation runs

This is optional and intentionally limited. It can be useful for checking only the best run, or a few selected runs, without spending too much.

In [14]:
RUN_BATCH_LLM_JUDGE = True
BATCH_LLM_JUDGE_ROOT = EVALUATIONS_ROOT / "batch_no_gold_llm_judge"
BATCH_JUDGE_MAX_ITEMS = 10
BATCH_JUDGE_MAX_WORKERS = 4
BATCH_COUNT_SUBJUDGE_PARSE_ERRORS = False
SELECTED_RUN_NAMES = []  # empty means first run only, or put names like ["layer12_from_l11"]

if RUN_BATCH_LLM_JUDGE:
    selected_states = layer12_states
    if SELECTED_RUN_NAMES:
        selected_states = [p for p in layer12_states if p.parent.parent.name in set(SELECTED_RUN_NAMES)]
    else:
        selected_states = layer12_states[:1]

    rows = []
    for state_path in selected_states:
        run_name = state_path.parent.parent.name
        out_dir = BATCH_LLM_JUDGE_ROOT / run_name
        cmd = [
            sys.executable, "-m", "neoolaf.evaluation", "no-gold",
            "--state", str(state_path),
            "--reference-ontology", str(SEED_ONTOLOGY),
            "--output", str(out_dir),
            "--llm-judge-panel",
            "--judge-model", JUDGE_MODEL,
            "--judge-max-items", str(BATCH_JUDGE_MAX_ITEMS),
            "--judge-max-workers", str(BATCH_JUDGE_MAX_WORKERS),
        ]
        if BATCH_COUNT_SUBJUDGE_PARSE_ERRORS:
            cmd.append("--count-subjudge-parse-errors")
        result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
        if result.returncode != 0:
            print("FAILED", run_name)
            print(result.stderr)
            continue
        with open(out_dir / "no_gold.summary.json", "r", encoding="utf-8") as f:
            s = json.load(f)
        lj = s.get("llm_judge_panel", s.get("llm_judge", {}))
        rows.append({"run": run_name, **lj})

    judge_batch_df = pd.DataFrame(rows)
    display(judge_batch_df)
else:
    print("Batch LLM judge disabled.")

,run,model,judged_count,valid_count,weak_count,invalid_count,inconclusive_count,parse_error_count,count_subjudge_parse_errors,average_score,supported_rate,relation_supported_rate,direction_correct_rate,high_agreement_count,medium_agreement_count,low_agreement_count
0,layer12_from_l11,openrouter/openai/gpt-oss-20b,10,10,0,0,0,0,False,0.947,1.0,1.0,1.0,10,0,0
